# 🖼️ 06 — CNN Image Classification (Clothing)

**E-Commerce Customer Behavior Analysis & Hybrid Recommendation System**

---

## Objective
Classify clothing images into categories using a custom CNN model built with PyTorch.

## Architecture
- 3 Convolutional blocks (Conv2d → BatchNorm → ReLU → MaxPool)
- Adaptive Average Pooling
- Fully connected classifier with Dropout
- Early stopping & learning rate scheduling

In [ ]:
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

DATA_DIR = r'E:\Projects\E-Commerce DM\data\raw\Clothes_Dataset'
MODEL_DIR = r'E:\Projects\E-Commerce DM\models'
OUTPUT_DIR = r'E:\Projects\E-Commerce DM\data\generated'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32; EPOCHS = 30; IMG_SIZE = 128; LEARNING_RATE = 0.001; PATIENCE = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

## 1. Load & Prepare Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR)
classes = full_dataset.classes
print(f'✅ {len(full_dataset)} images, {len(classes)} classes: {classes}')

class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset, self.indices, self.transform = dataset, indices, transform
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform: img = self.transform(img)
        return img, label
    def __len__(self): return len(self.indices)

gen = torch.Generator().manual_seed(42)
n = len(full_dataset); train_sz = int(0.70*n); val_sz = int(0.15*n); test_sz = n - train_sz - val_sz
indices = torch.randperm(n, generator=gen).tolist()
train_ds = TransformedSubset(full_dataset, indices[:train_sz], train_transform)
val_ds = TransformedSubset(full_dataset, indices[train_sz:train_sz+val_sz], eval_transform)
test_ds = TransformedSubset(full_dataset, indices[train_sz+val_sz:], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'📦 Train: {train_sz} | Val: {val_sz} | Test: {test_sz}')

## 2. Define CNN Model

In [ ]:
class ClothingCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4,4)))
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(64*4*4, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.features(x); x = x.view(x.size(0), -1); return self.classifier(x)

model = ClothingCNN(len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
print(model)

## 3. Training Loop

In [ ]:
print(f'🚀 Training for up to {EPOCHS} epochs (patience={PATIENCE})...')
start_time = time.time()
best_val_loss = float('inf'); patience_counter = 0
best_model_path = os.path.join(MODEL_DIR, 'cnn_clothing_best.pth')

for epoch in range(EPOCHS):
    model.train(); running_loss = correct = total = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()*inputs.size(0); _, pred = outputs.max(1)
        total += labels.size(0); correct += pred.eq(labels).sum().item()
    train_loss = running_loss/total; train_acc = 100.*correct/total

    model.eval(); val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs); loss = criterion(outputs, labels)
            val_loss += loss.item()*inputs.size(0); _, pred = outputs.max(1)
            val_total += labels.size(0); val_correct += pred.eq(labels).sum().item()
    val_loss /= val_total; val_acc = 100.*val_correct/val_total
    scheduler.step(val_loss)
    print(f'  Epoch {epoch+1}/{EPOCHS} -> Train: {train_loss:.4f}/{train_acc:.1f}% | Val: {val_loss:.4f}/{val_acc:.1f}%')

    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_counter = 0; torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  ⚡ Early stopping at epoch {epoch+1}'); break

print(f'✅ Training completed in {(time.time()-start_time):.1f}s')

## 4. Evaluation

In [ ]:
model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs); _, pred = outputs.max(1)
        all_preds.extend(pred.cpu().numpy()); all_labels.extend(labels.cpu().numpy())

print('📝 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=classes))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('CNN Confusion Matrix - Clothing Dataset')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cnn_confusion_matrix.png'))
plt.show()

torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'cnn_clothing.pth'))
print('✅ Model saved')